# Priority Classifier v3 — Approccio Ibrido (LinearSVC + LLM)

**ZHC — Zucchetti Healthcare | TicketClassifier**

## Strategia
Il LinearSVC v2 raggiunge già macro F1 = 0.8451.  
L'obiettivo non è sostituirlo, ma **rafforzarlo** sui casi incerti:

| Situazione | Approccio |
|---|---|
| decision_function margin alto | LinearSVC diretto (veloce, economico) |
| margin basso / ambiguità P1-P2 | LLM con regole PO16 + few-shot |
| P4 (rara, spesso confusa) | Sempre LLM come secondo parere |

## Features del ticket in input
- `oggetto` — titolo del ticket (sempre valorizzato)
- `descrizione` — testo pulito (99.8% coverage)
- `priorita_iniziale_cliente` — P1/P2/P3/P4 impostata dal cliente sul portale ZConnect

---

## STEP 1 — Import e configurazione percorsi

In [26]:
import os
import re
import json
import time
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Literal, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv

# ─── Percorsi di progetto ────────────────────────────────────────────────────
BASE_DIR  = Path(r'C:\Users\matteo.segatto\Desktop\TicketClassifier')
MODEL_DIR = BASE_DIR / 'modelli' / 'priority_v2'
DATA_DIR  = BASE_DIR / 'data'
EMB_DIR   = BASE_DIR / 'embeddings'
OUT_DIR   = BASE_DIR / 'data'

load_dotenv(BASE_DIR / '.env')

print('BASE_DIR:', BASE_DIR)
print('MODEL_DIR exists:', MODEL_DIR.exists())
print('API key OpenAI:', 'trovata' if os.getenv('OPENAI_API_KEY') else 'MANCANTE')

# Se usi Claude invece di OpenAI:
# print('API key Anthropic:', 'trovata' if os.getenv('ANTHROPIC_API_KEY') else 'MANCANTE')

BASE_DIR: C:\Users\matteo.segatto\Desktop\TicketClassifier
MODEL_DIR exists: True
API key OpenAI: trovata


## STEP 2 — Carica modello LinearSVC v2 e metadata

In [27]:
# Carica il classificatore SVC già addestrato
svc = joblib.load(MODEL_DIR / 'classificatore_svc.pkl')

with open(MODEL_DIR / 'metadata.json') as f:
    meta = json.load(f)

CLASSI           = meta['classi']           # ['P1', 'P2', 'P3', 'P4']
KEYWORD_URGENZA  = meta['keyword_urgenza']  # ['urgente', 'immediato', ...]
EMBEDDING_MODEL  = meta['modello_embedding']  # 'intfloat/multilingual-e5-base'
E5_PREFIX        = meta['prefisso_e5']       # 'query: '

print('Modello caricato:', type(svc).__name__)
print('Classi:', CLASSI)
print('Macro F1 su test set:', meta['macro_f1_test'])
print('Accuracy su test set:', meta['accuracy_test'])

Modello caricato: LinearSVC
Classi: ['P1', 'P2', 'P3', 'P4']
Macro F1 su test set: 0.8413
Accuracy su test set: 0.8841


## STEP 3 — Carica encoder embedding (E5) e OHE priorità

Il modello usa 2 feature concatenate:
1. Embedding E5 768d (testo ticket)
2. One-hot encoding priorità iniziale cliente (4 valori: P1/P2/P3/P4)


In [28]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import OneHotEncoder
import scipy.sparse as sp

# ─── Modello embedding ───────────────────────────────────────────────────────
print('Carico E5 (può richiedere qualche secondo)...')
e5_model = SentenceTransformer(EMBEDDING_MODEL)
print('E5 caricato.')

# ─── OHE priorità iniziale cliente ──────────────────────────────────────────
# Ricarica lo stesso encoder usato in training.
# Se il file non esiste, lo ricostruiamo con le stesse categorie.
ohe_path = MODEL_DIR / 'ohe_encoder.pkl'
if ohe_path.exists():
    ohe = joblib.load(ohe_path)
    print('OHE caricato da file.')
else:
    ohe = OneHotEncoder(
        categories=[['P1', 'P2', 'P3', 'P4']],
        sparse_output=True,
        handle_unknown='ignore'
    )
    # Fit sulle 4 categorie fisse (non serve dataset)
    ohe.fit([['P1'], ['P2'], ['P3'], ['P4']])
    print('OHE ricreato (file non trovato — ok se usi le stesse categorie).')

def build_features(oggetto: str,
                   descrizione: str,
                   priorita_iniziale: str = 'P3') -> np.ndarray:
    """Costruisce il vettore feature per un singolo ticket."""
    # 1. Testo completo
    testo = f"{oggetto or ''} {descrizione or ''}".strip()

    # 2. Embedding E5 (con prefisso 'query: ')
    emb = e5_model.encode(
        E5_PREFIX + testo,
        normalize_embeddings=True
    ).reshape(1, -1)  # shape (1, 768)

    # 3. OHE priorità iniziale cliente
    p_init = priorita_iniziale if priorita_iniziale in ['P1','P2','P3','P4'] else 'P3'
    ohe_vec = ohe.transform([[p_init]])  # sparse (1, 4)

    # 4. Concatenazione: [768d | 4d] = 772d (stesso schema del training)
    X = np.hstack([
        emb,
        ohe_vec.toarray()
    ])
    return X  # shape (1, 772)

# Test rapido
X_test = build_features(
    'Il sistema non funziona',
    'La piattaforma è completamente inaccessibile da questa mattina',
    priorita_iniziale='P1'
)
print('Feature shape:', X_test.shape)  # deve essere (1, 772)

Carico E5 (può richiedere qualche secondo)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


E5 caricato.
OHE caricato da file.
Feature shape: (1, 772)


c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


## STEP 4 — Classificatore SVC con confidenza calibrata

`LinearSVC` non ha `.predict_proba()`, ma ha `.decision_function()` che restituisce
la distanza dal margine. Usiamo la distanza per stimare la confidenza:
- **margin alto** → il modello è sicuro → output diretto
- **margin basso** → ambiguità → passiamo all'LLM

Normalizziamo con softmax sui decision scores per avere una pseudo-probabilità
confrontabile con l'output strutturato dell'LLM.

In [29]:
from scipy.special import softmax

def predict_svc_with_confidence(X: np.ndarray) -> tuple[str, float, dict]:
    """
    Predice la priorità con il LinearSVC e stima la confidenza.

    Returns:
        classe_predetta: 'P1', 'P2', 'P3' o 'P4'
        confidenza:      float in [0, 1] — pseudo-probabilità softmax
        probs:           dict con pseudo-prob per ogni classe
    """
    scores = svc.decision_function(X)[0]  # shape (4,) — una per classe
    probs  = softmax(scores * 2)          # * 2 per accentuare il picco (calibrazione empirica)

    best_idx    = np.argmax(probs)
    classe      = CLASSI[best_idx]
    confidenza  = float(probs[best_idx])
    probs_dict  = {c: float(p) for c, p in zip(CLASSI, probs)}

    return classe, confidenza, probs_dict


# Test rapido
classe, conf, probs = predict_svc_with_confidence(X_test)
print(f'Predizione SVC: {classe} (confidenza: {conf:.1%})')
print('Distribuzione classi:', {k: f'{v:.1%}' for k, v in probs.items()})

Predizione SVC: P1 (confidenza: 97.1%)
Distribuzione classi: {'P1': '97.1%', 'P2': '1.1%', 'P3': '0.8%', 'P4': '1.0%'}


## STEP 5 — Definizioni PO16 per il prompt LLM

Queste sono le definizioni **ufficiali** dalla procedura PO16 (Procedura Assistenza SW).
Le usiamo come "ground truth" per il reasoning dell'LLM — il modello non deve inventarsi
le regole, le trova direttamente nel prompt.

In [30]:
# ─── Definizioni PO16 (versione con focus operativo) ─────────────────────────────
PRIORITY_DEFINITIONS = """
P1 — Molto Alta (Blocco Totale):
  Blocco completo e inaccessibilità all'intera piattaforma e relativa operatività.
  TUTTI gli utenti della struttura non riescono a lavorare.
  TUTTI i moduli/funzioni sono inaccessibili.
  Si escludono problematiche infrastrutturali esterne (VPN, rete cliente, PC).

P2 — Alta (Blocco Funzionale senza Workaround):
  Blocco di funzionalità essenziali non differibili per cui non sia possibile
  procedere con metodi alternativi (es. terapie, fatturazione, paghe).
  La piattaforma è accessibile ma UNA funzione critica è inutilizzabile.
  Anche se molti utenti sono coinvolti, la piattaforma rimane parzialmente funzionante.

P3 — Media (Malfunzionamento con Workaround):
  Malfunzionamento di funzioni essenziali ma gestibile con metodi alternativi
  o supporto operativo. Include: domande procedurali, configurazioni,
  personalizzazioni di elenchi, problemi su singoli utenti/dispositivi.
  Esiste sempre un modo alternativo per portare avanti il lavoro.

P4 — Bassa (Impatto Minimo):
  Malfunzionamento che non ostacola il regolare svolgimento del lavoro.
  Richieste evolutive, miglioramenti estetici, anomalie visive con workaround
  immediato, correzioni di dati inseriti erroneamente dall'utente.
"""

# ─── Regole di disambiguazione (applica prima di classificare) ──────────────────
DISAMBIGUATION_RULES = """
REGOLA 1 — P1 richiede impatto TOTALE:
  P1 si assegna SOLO se TUTTI gli utenti della struttura non riescono a lavorare
  E TUTTA la piattaforma è inaccessibile.
  Se anche un solo reparto, modulo o utente riesce a operare → non è P1.
  Se il problema è circoscritto a un modulo, un report, o un gruppo di utenti → P2 al massimo.

REGOLA 2 — Segnali linguistici fuorvianti (ignora ai fini della priorità):
  Maiuscole nell'oggetto ("URGENTE", "BLOCCATI", "AIUTO"), punti esclamativi
  multipli, parole come "urgente", "prioritario", "scadenza oggi", "emergenza"
  NON determinano la priorità tecnica. Valuta SOLO l'impatto operativo reale.

REGOLA 3 — Priorità impostata dal cliente = segnale debole:
  I clienti tendono sistematicamente a sovra-esagerare la priorità, in particolare
  impostando P1 anche per problemi circoscritti a un modulo o pochi utenti.
  La priorità cliente è un indizio, non una prova. Il testo descrittivo conta di più.

REGOLA 4 — Principio di cautela P1 vs P2:
  In caso di dubbio tra P1 e P2, scegli P2.
  La de-escalation da P1 a P2 è quasi sempre preferibile alla escalation non giustificata.
  Un P1 sbagliato mobilita risorse di emergenza in modo inappropriato.
  Il dubbio P1/P2 si risolve verificando l'ampiezza del blocco: se il testo dice esplicitamente che TUTTI gli utenti sono bloccati sull'INTERA piattaforma assegna P1, altrimenti P2. Non abbassare automaticamente la priorità.

REGOLA 5 — Principio di cautela P2 vs P3:
  In caso di dubbio tra P2 e P3, chiedi: esiste un workaround pratico?
  Se il cliente può portare avanti il lavoro con un metodo alternativo (anche manuale)
  → P3. Il workaround non deve essere comodo, deve essere praticabile.
"""

# ─── Esempi few-shot — asimmetrici, mirati sui casi borderline ───────────────────────
# Struttura: 2× (sembra P1 → è P2) + 2× (sembra P2 → è P3) + 1× P1 vero + 1× P4
FEW_SHOT_EXAMPLES = [
    # ── P1 vero: blocco totale multi-dispositivo ──────────────────────────────
    {
        "ticket":            "229391",
        "oggetto":           "IMPOSSIBILITA' D'ACCESSO",
        "descrizione":       ("Da questa mattina dalle ore 6:00 circa non riusciamo ad accedere "
                              "a CBA, sia dal PC che dai tablet. Non riusciamo a scrivere le "
                              "consegne né ad accedere ad alcun modulo."),
        "priorita_cliente":  "P1",
        "priorita_corretta": "P1",
        "reasoning":         ("Fermo operativo totale su tutti i dispositivi e tutti i moduli. "
                              "Corrisponde esattamente alla definizione P1 della procedura PO16."),
    },

# ── P1 vero: blocco login, tono sobrio ───────────────────────────────────
    {
        "ticket":            "270023",
        "oggetto":           "CBA NON FUNZIONANTE",
        "descrizione":       ("Buongiorno si comunica che il programma CBA non funziona "
                              "e non si riesce ad eseguire il login."),
        "priorita_cliente":  "P1",
        "priorita_corretta": "P1",
        "reasoning":         ("Tono sobrio ma il login è la porta d'accesso a tutti i moduli. "
                              "Impossibilità di accedere = blocco totale dell'operatività → P1 "
                              "anche senza linguaggio allarmistico."),
    },
    # ── P1 vero: inaccessibilità prolungata, descrizione tecnica ─────────────
    {
        "ticket":            "264806",
        "oggetto":           "INACCESSIBILITA'",
        "descrizione":       ("Buongiorno, da questa notte (alle ore 00:30 circa) non c'è "
                              "nessuna accessibilità al programma."),
        "priorita_cliente":  "P1",
        "priorita_corretta": "P1",
        "reasoning":         ("Totale mancanza di accesso alla piattaforma da diverse ore. "
                              "Nessun riferimento a singoli moduli o record: il problema "
                              "colpisce l'intera piattaforma → P1."),
    },

    # ── P2: linguaggio urgente ma blocco su singola funzione ─────────────────
    {
        "ticket":            "230917",
        "oggetto":           "STAMPA BILANCIO",
        "descrizione":       ("Buongiorno, non si riesce a stampare il bilancio, in quanto "
                              "segnala errore relativamente ad una fattura. "
                              "Richiedo collegamento telefonico."),
        "priorita_cliente":  "P1",
        "priorita_corretta": "P2",
        "reasoning":         ("Il cliente usa tono urgente ma il problema è circoscritto a un "
                              "singolo report/fattura. La piattaforma è accessibile. "
                              "Blocco funzionale parziale → P2, non P1."),
    },
    # ── P2: parola 'urgenza' ma impatto su sotto-modulo ──────────────────────
    {
        "ticket":            "227260",
        "oggetto":           "Servizi accessori Rsa",
        "descrizione":       ("Buongiorno, non riesco ad elaborare i servizi accessori rsa "
                              "nel mese in corso. Vi prego di chiamare con urgenza "
                              "nella giornata odierna."),
        "priorita_cliente":  "P1",
        "priorita_corretta": "P2",
        "reasoning":         ("Il blocco riguarda solo l'elaborazione dei servizi accessori, "
                              "non l'intera piattaforma. L'operatività core della struttura "
                              "non è compromessa → P2."),
    },
    # ── P3: errore procedurale utente con workaround ──────────────────────────
    {
        "ticket":            "214168",
        "oggetto":           "Chiusura cartella",
        "descrizione":       ("Un ospite è rientrato dall'ospedale e per errore invece di "
                              "chiudere il ricovero ospedaliero nelle 'assenze', è stata "
                              "chiusa la cartella nei 'ricoveri'. Ho necessità di ripristinare "
                              "il fascicolo."),
        "priorita_cliente":  "P2",
        "priorita_corretta": "P3",
        "reasoning":         ("Errore procedurale dell'utente risolvibile con supporto operativo. "
                              "Non blocca le funzionalità cliniche essenziali → P3."),
    },
    # ── P3: richiesta configurazione, nessun blocco ───────────────────────────
    {
        "ticket":            "226495",
        "oggetto":           "Raggruppamento contabile",
        "descrizione":       ("Salve, abbiamo assunto un fisioterapista per la prima volta come "
                              "dipendente, non riesco inserire nei raggruppamenti contabili "
                              "la sua figura."),
        "priorita_cliente":  "P2",
        "priorita_corretta": "P3",
        "reasoning":         ("Richiesta di istruzioni per configurare una nuova anagrafica. "
                              "Nessun bug o blocco tecnico presente → P3."),
    },
    # ── P4: richiesta estetica senza impatto operativo ────────────────────────
    {
        "ticket":            "250420",
        "oggetto":           "personalizzazione etichette",
        "descrizione":       ("Ciao, si possono personalizzare i campi delle etichette? "
                              "Ad esempio inserire la foto dell'ospite."),
        "priorita_cliente":  "P3",
        "priorita_corretta": "P4",
        "reasoning":         ("Richiesta di miglioramento estetico senza alcun impatto "
                              "sull'operatività. Nessun malfunzionamento → P4."),
    },
]

print(f"Definizioni PO16: {len(PRIORITY_DEFINITIONS.strip().splitlines())} righe")
print(f"Regole disambiguazione: {len(DISAMBIGUATION_RULES.strip().splitlines())} righe")
print(f"Esempi few-shot: {len(FEW_SHOT_EXAMPLES)}")
print(f"  - Sembra P1 \u2192 \u00e8 P2: {sum(1 for e in FEW_SHOT_EXAMPLES if e['priorita_cliente']=='P1' and e['priorita_corretta']=='P2')}")
print(f"  - Sembra P2 \u2192 \u00e8 P3: {sum(1 for e in FEW_SHOT_EXAMPLES if e['priorita_cliente']=='P2' and e['priorita_corretta']=='P3')}")
print(f"  - P1 vero:           {sum(1 for e in FEW_SHOT_EXAMPLES if e['priorita_corretta']=='P1')}")
print(f"  - P4:                {sum(1 for e in FEW_SHOT_EXAMPLES if e['priorita_corretta']=='P4')}")


Definizioni PO16: 22 righe
Regole disambiguazione: 26 righe
Esempi few-shot: 8
  - Sembra P1 → è P2: 2
  - Sembra P2 → è P3: 2
  - P1 vero:           3
  - P4:                1


## STEP 6 — Classificatore LLM con output strutturato (Pydantic)

Usiamo lo stesso pattern del `FewShot_ExampleExtractor` per l'area classifier:
- **Pydantic** per output strutturato garantito
- **Temperatura 0** per riproducibilità
- Il LLM restituisce: `priorita`, `confidenza`, `reasoning`

Funziona sia con OpenAI (GPT-4o-mini) che con Anthropic (Claude Haiku).

In [31]:
from openai import OpenAI

client_llm = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ─── Schema output strutturato ─────────────────────────────────────────────
PrioritaType = Literal["P1", "P2", "P3", "P4"]

class ClassificazionePriorita(BaseModel):
    priorita:   PrioritaType = Field(description="Priorità assegnata: P1, P2, P3 o P4")
    confidenza: float        = Field(ge=0.0, le=1.0, description="Confidenza da 0 a 1")
    reasoning:  str          = Field(description="Spiegazione breve (max 2 frasi)")


# ─── Costruzione system prompt ────────────────────────────────────────────
def build_system_prompt() -> str:
    """Costruisce il system prompt con definizioni PO16, regole disambiguazione e few-shot reali."""

    esempi_str = ""
    for ex in FEW_SHOT_EXAMPLES:
        esempi_str += (
            f"\n---"
            f"\nTESTO TICKET: {ex['oggetto']}. {ex['descrizione']}"
            f"\nPRIORITÀ CLIENTE: {ex['priorita_cliente']}"
            f"\n\u2192 PRIORITÀ CORRETTA: {ex['priorita_corretta']}"
            f"\nREASONING: {ex['reasoning']}\n"
        )

    return f"""Ricevi solo i ticket sui quali il modello automatico è incerto. Il testo può contenere segnali fuorvianti. Valuta l'IMPATTO REALE.

Sei un assistente specializzato nella classificazione di ticket di assistenza software per Zucchetti Healthcare (ZHC). Il tuo compito è assegnare la priorità corretta a ciascun ticket basandoti sulle definizioni ufficiali della procedura PO16.

## DEFINIZIONI PRIORITÀ (da procedura PO16)
{PRIORITY_DEFINITIONS}

## REGOLE DI DISAMBIGUAZIONE
{DISAMBIGUATION_RULES}

## ESEMPI REALI DAL DATASET ZHC
(Nota: la maggioranza degli esempi mostra casi borderline dove la priorità cliente è sbagliata.
Questo è intenzionale: sono i casi più frequenti che ti verranno sottoposti.)
{esempi_str}
Rispondi SOLO con l'oggetto JSON strutturato richiesto. Nessun testo aggiuntivo."""


SYSTEM_PROMPT = build_system_prompt()
print("System prompt generato.")
print(f"Lunghezza: {len(SYSTEM_PROMPT.split())} parole")
print(f"Esempi inclusi: {SYSTEM_PROMPT.count('PRIORIT\u00c0 CORRETTA')}")


System prompt generato.
Lunghezza: 950 parole
Esempi inclusi: 8


In [32]:
def preprocess_testo(oggetto: str, descrizione: str, max_parole: int = 300) -> str:
    """Pulisce e tronca il testo prima di inviarlo all'LLM."""
    testo = f"{oggetto or ''} {descrizione or ''}"
    # Rimuovi HTML
    testo = re.sub(r'<[^>]+>', ' ', testo)
    # Rimuovi firme email e dati di contatto
    testo = re.sub(r'Orario reperibilit.*', '', testo, flags=re.DOTALL | re.IGNORECASE)
    testo = re.sub(r'Cordiali saluti.*',    '', testo, flags=re.DOTALL | re.IGNORECASE)
    testo = re.sub(r'Telefono:?\s*[\d\s+]+', '', testo, flags=re.IGNORECASE)
    # Normalizza spazi
    testo = re.sub(r'\s+', ' ', testo).strip()
    # Tronca
    parole = testo.split()
    if len(parole) > max_parole:
        testo = ' '.join(parole[:max_parole]) + '...'
    return testo


def classifica_priorita_llm(oggetto: str,
                             descrizione: str,
                             priorita_iniziale_cliente: str = 'P3') -> ClassificazionePriorita:
    """Classifica la priorità di un ticket usando GPT-4o-mini con few-shot + PO16."""
    testo_pulito = preprocess_testo(oggetto, descrizione)

    user_msg = (
        f'OGGETTO: {oggetto}\n'
        f'DESCRIZIONE: {testo_pulito}\n'
        f'PRIORITÀ IMPOSTATA DAL CLIENTE: {priorita_iniziale_cliente}'
    )

    response = client_llm.beta.chat.completions.parse(
        model='gpt-4o-mini',
        temperature=0,
        max_tokens=150,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_msg},
        ],
        response_format=ClassificazionePriorita,
    )
    return response.choices[0].message.parsed


# ─── Test rapido ─────────────────────────────────────────────────────────────
print('Test LLM su esempio P1 reale...')
risultato = classifica_priorita_llm(
    'SISTEMA COMPLETAMENTE BLOCCATO',
    'Da stamattina nessuno riesce ad entrare nel gestionale. '
    'Abbiamo provato con tutti i PC e tutti gli utenti. '
    'La struttura è completamente ferma, urgente!',
    priorita_iniziale_cliente='P1'
)
print(f'  Priorità:   {risultato.priorita}')
print(f'  Confidenza: {risultato.confidenza:.0%}')
print(f'  Reasoning:  {risultato.reasoning}')

Test LLM su esempio P1 reale...
  Priorità:   P1
  Confidenza: 100%
  Reasoning:  Il ticket descrive un blocco totale dell'accesso al gestionale per tutti gli utenti, confermando l'impossibilità di operare su tutta la piattaforma. Corrisponde esattamente alla definizione P1 della procedura PO16.


## STEP 7 — Classificatore Ibrido

La classe `HybridPriorityClassifier` combina i due approcci:

```
ticket → LinearSVC → confidenza > soglia? → output diretto
                   →       no           → LLM → output con reasoning
```

**Soglia consigliata**: 0.75 — da calibrare sul validation set.
- Sopra 0.75: ~80% dei ticket → SVC diretto (veloce, gratis)
- Sotto 0.75: ~20% dei ticket → LLM (~0.5 ms latenza, ~$0.0002/ticket)

In [33]:
from dataclasses import dataclass

@dataclass
class PredizionePriorita:
    """Risultato finale del classificatore ibrido."""
    priorita:    str
    confidenza:  float
    metodo:      str   # 'svc' oppure 'llm'
    reasoning:   Optional[str] = None  # disponibile solo se metodo='llm'
    probs_svc:   Optional[dict] = None  # distribuzione SVC (debug)


class HybridPriorityClassifier:
    """
    Classificatore ibrido LinearSVC + LLM per la priorità dei ticket ZHC.

    Parameters
    ----------
    soglia_confidenza : float
        Soglia sotto la quale il SVC cede il passo all'LLM.
        Applicata SOLO quando il SVC predice P1: se conf_svc < soglia e classe == 'P1',
        il ticket va all'LLM. Per P2/P3/P4 il SVC è sempre usato direttamente.
        Default: 0.75
    """

    def __init__(self, soglia_confidenza: float = 0.75):
        self.soglia_confidenza = soglia_confidenza

    def predict(self,
                oggetto:                  str,
                descrizione:              str,
                priorita_iniziale_cliente: str = 'P3') -> PredizionePriorita:
        """
        Classifica la priorità di un ticket.

        1. Estrae le feature e interroga il LinearSVC
        2. Se il SVC predice P1 con confidenza bassa → usa LLM
        3. Per P2/P3/P4 incerti il SVC è più affidabile dell'LLM → output diretto
        4. Restituisce il risultato con metadati sul percorso usato
        """
        # ── Fast path: LinearSVC ──────────────────────────────────────────────
        X = build_features(oggetto, descrizione, priorita_iniziale_cliente)
        classe_svc, conf_svc, probs_svc = predict_svc_with_confidence(X)

        # Manda all'LLM SOLO se il SVC predice P1 con bassa confidenza.
        # Per P2/P3/P4 incerti il SVC è comunque più affidabile dell'LLM.
        usa_svc = not (classe_svc == 'P1' and conf_svc < self.soglia_confidenza)

        if usa_svc:
            return PredizionePriorita(
                priorita   = classe_svc,
                confidenza = conf_svc,
                metodo     = 'svc',
                probs_svc  = probs_svc,
            )

        # ── Slow path: LLM ───────────────────────────────────────────────────
        llm_result = classifica_priorita_llm(oggetto, descrizione, priorita_iniziale_cliente)

        return PredizionePriorita(
            priorita   = llm_result.priorita,
            confidenza = llm_result.confidenza,
            metodo     = 'llm',
            reasoning  = llm_result.reasoning,
            probs_svc  = probs_svc,  # conserviamo i score SVC per debug
        )


# Istanzia il classificatore
clf = HybridPriorityClassifier(soglia_confidenza=0.75)
print('Classificatore ibrido pronto.')
print(f'Soglia confidenza: {clf.soglia_confidenza:.0%}')
print(f'Routing LLM: solo P1 con conf < {clf.soglia_confidenza:.0%}')


Classificatore ibrido pronto.
Soglia confidenza: 75%
Routing LLM: solo P1 con conf < 75%


## STEP 8 — Test su casi reali

Testiamo sui 4 pattern più comuni dal dataset (vedi EDA.ipynb):
- P1 → P2: 6.129 correzioni (cliente esagera l'urgenza)
- P3 → P2: 4.190 correzioni (cliente sottostima)
- P1 → P3: 2.648 correzioni (domanda ordinaria con P1 per errore)
- P4: rara, spesso confusa con P3

In [34]:
CASI_TEST = [
    {
        'descrizione_test': 'P1→P2: cliente esagera, blocco modulo non totale',
        'oggetto':          'MODULO PRESENZE BLOCCATO - URGENTISSIMO',
        'descrizione':      'Il modulo Rilevazione Presenze non carica le timbrature di ieri. '
                            'Tutti gli altri moduli funzionano normalmente. '
                            'Abbiamo la chiusura mese domani, è urgentissimo!',
        'p_cliente':        'P1',
        'p_attesa':         'P2',
    },
    {
        'descrizione_test': 'P1→P3: domanda operativa mascherata da urgenza',
        'oggetto':          'Non trovo la funzione stampa cedolini',
        'descrizione':      'Buongiorno, cercavo come si fa a stampare i cedolini in PDF '
                            'dal modulo Gestione Stipendi. Non riesco a trovare il tasto. '
                            'Grazie',
        'p_cliente':        'P1',
        'p_attesa':         'P3',
    },
    {
        'descrizione_test': 'P3→P2: cliente sottostima, bug contabile critico',
        'oggetto':          'Errore calcolo IVA su alcune fatture',
        'descrizione':      'Abbiamo notato che alcune fatture emesse questa settimana '
                            'hanno il calcolo IVA errato. Circa il 30% delle fatture '
                            'è sbagliato. Dobbiamo inviare le fatture elettroniche entro domani.',
        'p_cliente':        'P3',
        'p_attesa':         'P2',
    },
    {
        'descrizione_test': 'P4: richiesta evolutiva',
        'oggetto':          'Richiesta aggiunta campo note in scheda paziente',
        'descrizione':      'Sarebbe comodo avere un campo note aggiuntivo nella '
                            'scheda anagrafica del paziente per inserire informazioni '
                            'aggiuntive non previste dai campi standard.',
        'p_cliente':        'P2',
        'p_attesa':         'P4',
    },
]

print('=' * 70)
for caso in CASI_TEST:
    pred = clf.predict(
        caso['oggetto'],
        caso['descrizione'],
        caso['p_cliente']
    )
    ok = '✓' if pred.priorita == caso['p_attesa'] else '✗'
    print(f"{ok} [{caso['descrizione_test']}]")
    print(f"   Attesa: {caso['p_attesa']} | Cliente: {caso['p_cliente']} | "
          f"Predetto: {pred.priorita} ({pred.confidenza:.0%}) via {pred.metodo.upper()}")
    if pred.reasoning:
        print(f"   Reasoning: {pred.reasoning}")
    print()

✗ [P1→P2: cliente esagera, blocco modulo non totale]
   Attesa: P2 | Cliente: P1 | Predetto: P1 (99%) via SVC

✗ [P1→P3: domanda operativa mascherata da urgenza]
   Attesa: P3 | Cliente: P1 | Predetto: P1 (81%) via SVC

✗ [P3→P2: cliente sottostima, bug contabile critico]
   Attesa: P2 | Cliente: P3 | Predetto: P3 (96%) via SVC

✗ [P4: richiesta evolutiva]
   Attesa: P4 | Cliente: P2 | Predetto: P2 (86%) via SVC



c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


## STEP 9 — Valutazione su campione del test set

Carichiamo il dataset pulito e valutiamo il classificatore ibrido su un campione stratificato,
confrontando con il solo LinearSVC v2.

**Nota**: usa un campione piccolo (100-200 ticket) per il LLM — controlla i costi prima!

In [35]:
from sklearn.metrics import classification_report, f1_score

# ─── Carica dataset pulito ────────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'dataset_clean.csv', parse_dates=['data_creazione'])
print(f'Dataset: {len(df):,} righe')

# Filtro test temporale (stesso split usato in training)
SOGLIA_SPLIT = pd.Timestamp('2025-11-01')
df_test = df[df['data_creazione'] >= SOGLIA_SPLIT].copy()
print(f'Test set (dal {SOGLIA_SPLIT.date()}): {len(df_test):,} righe')
print('\nDistribuzione priorità nel test set:')
print(df_test['priorita_finale'].value_counts())

Dataset: 61,156 righe
Test set (dal 2025-11-01): 14,809 righe

Distribuzione priorità nel test set:
priorita_finale
P3    8523
P2    4087
P1    1880
P4     319
Name: count, dtype: int64


In [40]:
# Campionamento stratificato (compatibile pandas 2.x)
# IMPORTANTE: bilancia il campione per avere abbastanza esempi di P4
N_PER_CLASSE = 100  # 100 * 4 classi = 400 ticket totali

campioni = pd.concat([
    df_test[df_test["priorita_finale"] == cls].sample(
        min(N_PER_CLASSE, (df_test["priorita_finale"] == cls).sum()),
        random_state=42
    )
    for cls in CLASSI
    if (df_test["priorita_finale"] == cls).sum() > 0
]).reset_index(drop=True)

print(f"Campione totale: {len(campioni)} ticket")
print(campioni["priorita_finale"].value_counts())

Campione totale: 400 ticket
priorita_finale
P1    100
P2    100
P3    100
P4    100
Name: count, dtype: int64


In [41]:
# import per eliminare warning
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openai")
# ─── Valutazione ─────────────────────────────────────────────────────────────
risultati = []
errori    = 0

for i, (_, row) in enumerate(campioni.iterrows()):
    try:
        pred = clf.predict(
            str(row.get('testo_input', '')),
            '',
            str(row.get('priorita_iniziale_cliente', 'P3'))
        )
        risultati.append({
            'priorita_reale':    row['priorita_finale'],
            'priorita_predetta': pred.priorita,
            'confidenza':        pred.confidenza,
            'metodo':            pred.metodo,
            'reasoning':         pred.reasoning or '',
            'corretto':          row['priorita_finale'] == pred.priorita,
        })

        if (i + 1) % 20 == 0:
            acc = sum(r['corretto'] for r in risultati) / len(risultati)
            n_llm = sum(1 for r in risultati if r['metodo'] == 'llm')
            print(f'[{i+1}/{len(campioni)}] Accuracy: {acc:.1%} | '
                  f'LLM usato: {n_llm}/{len(risultati)} ({n_llm/len(risultati):.0%})')

        # Rate limiting
        time.sleep(0.1)

    except Exception as e:
        errori += 1
        print(f'  Errore su idx {i}: {e}')

df_ris = pd.DataFrame(risultati)

print('\n' + '=' * 60)
print('RISULTATI CLASSIFICATORE IBRIDO v3')
print('=' * 60)
print(classification_report(
    df_ris['priorita_reale'],
    df_ris['priorita_predetta'],
    zero_division=0
))
print(f'Macro F1 ibrido:  {f1_score(df_ris["priorita_reale"], df_ris["priorita_predetta"], average="macro", zero_division=0):.4f}')
print(f'Macro F1 SVC v2:  {meta["macro_f1_test"]}  (benchmark)')
print(f'\nLLM usato:        {(df_ris["metodo"]=="llm").sum()} / {len(df_ris)} ticket '
      f'({(df_ris["metodo"]=="llm").mean():.0%})')
print(f'Errori API:       {errori}')

c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarnin

[20/400] Accuracy: 80.0% | LLM usato: 4/20 (20%)


c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarnin

[40/400] Accuracy: 82.5% | LLM usato: 5/40 (12%)


c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarnin

...

[360/400] Accuracy: 83.3% | LLM usato: 35/360 (10%)


c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarnin

[380/400] Accuracy: 82.9% | LLM usato: 35/380 (9%)


c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\matteo.segatto\Desktop\TicketClassifier\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarnin

[400/400] Accuracy: 82.8% | LLM usato: 36/400 (9%)

RISULTATI CLASSIFICATORE IBRIDO v3
              precision    recall  f1-score   support

          P1       0.89      0.79      0.84       100
          P2       0.82      0.80      0.81       100
          P3       0.75      0.95      0.84       100
          P4       0.88      0.77      0.82       100

    accuracy                           0.83       400
   macro avg       0.84      0.83      0.83       400
weighted avg       0.84      0.83      0.83       400

Macro F1 ibrido:  0.8270
Macro F1 SVC v2:  0.8413  (benchmark)

LLM usato:        36 / 400 ticket (9%)
Errori API:       0


## STEP 10 — Analisi degli errori

Dove sbaglia ancora il classificatore ibrido?

In [42]:
# ─── Analisi errori per classe ────────────────────────────────────────────────
errori_df = df_ris[~df_ris['corretto']].copy()
print(f'Errori totali: {len(errori_df)} / {len(df_ris)} '
      f'({len(errori_df)/len(df_ris):.1%})')
print('\nErrori per classe reale:')
print(errori_df['priorita_reale'].value_counts())

print('\nMatrice confusione:')
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(
    df_ris['priorita_reale'],
    df_ris['priorita_predetta'],
    labels=CLASSI
)
cm_df = pd.DataFrame(cm, index=CLASSI, columns=CLASSI)
cm_df.index.name = 'Reale \ Predetto'
print(cm_df)

Errori totali: 69 / 400 (17.2%)

Errori per classe reale:
priorita_reale
P4    23
P1    21
P2    20
P3     5
Name: count, dtype: int64

Matrice confusione:
                  P1  P2  P3  P4
Reale \ Predetto                
P1                79  11   3   7
P2                 9  80   9   2
P3                 0   3  95   2
P4                 1   3  19  77


<>:16: SyntaxWarning: invalid escape sequence '\ '
<>:16: SyntaxWarning: invalid escape sequence '\ '
C:\Users\matteo.segatto\AppData\Local\Temp\ipykernel_13352\4161959428.py:16: SyntaxWarning: invalid escape sequence '\ '
  cm_df.index.name = 'Reale \ Predetto'


In [43]:
# ─── Dove ha aiutato l'LLM? ──────────────────────────────────────────────────
print('Performance SVC alone vs LLM alone (sui ticket che hanno passato l\'LLM):')

# Per i ticket dove il SVC era incerto e ha passato all'LLM:
df_llm_cases = df_ris[df_ris['metodo'] == 'llm'].copy()
if len(df_llm_cases) > 0:
    print(f'\nTicket gestiti da LLM: {len(df_llm_cases)}')
    print(f'Accuracy LLM su questi ticket: '
          f'{df_llm_cases["corretto"].mean():.1%}')
    print('\nDistribuzione classi reali nei casi LLM:')
    print(df_llm_cases['priorita_reale'].value_counts())

# ─── Esporta risultati ────────────────────────────────────────────────────────
output_path = OUT_DIR / 'hybrid_v3_risultati.csv'
df_ris.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'\nRisultati salvati in: {output_path}')

Performance SVC alone vs LLM alone (sui ticket che hanno passato l'LLM):

Ticket gestiti da LLM: 36
Accuracy LLM su questi ticket: 22.2%

Distribuzione classi reali nei casi LLM:
priorita_reale
P1    17
P2    13
P3     4
P4     2
Name: count, dtype: int64

Risultati salvati in: C:\Users\matteo.segatto\Desktop\TicketClassifier\data\hybrid_v3_risultati.csv


In [44]:
df_ris = pd.read_csv(OUT_DIR / 'hybrid_v3_risultati.csv')
df_llm = df_ris[df_ris['metodo'] == 'llm']

print("── Cosa predice il SVC sui ticket andati all'LLM ──")
print(df_llm['pred_svc'].value_counts() if 'pred_svc' in df_llm.columns 
      else "colonna pred_svc non salvata")

print("\n── Cosa predice l'LLM su questi ticket ──")
print(df_llm.groupby(['priorita_reale','priorita_predetta']).size())

── Cosa predice il SVC sui ticket andati all'LLM ──
colonna pred_svc non salvata

── Cosa predice l'LLM su questi ticket ──
priorita_reale  priorita_predetta
P1              P2                   7
                P3                   3
                P4                   7
P2              P2                   4
                P3                   7
                P4                   2
P3              P3                   2
                P4                   2
P4              P4                   2
dtype: int64


## STEP 11 — Prossimi step consigliati

### A. Migliorare il few-shot (priorità immediata)
Sostituire gli esempi manuali con esempi estratti automaticamente dal dataset,
usando il `decision_function` del LinearSVC per trovare i più prototipici per classe
(stesso approccio del `FewShot_ExampleExtractor.ipynb` per l'area).

```python
# Esempio: estrai i 3 ticket P4 con margine più alto
df_p4 = df_train[df_train['priorita_finale'] == 'P4']
X_p4 = build_features_batch(df_p4)  # da implementare
scores_p4 = svc.decision_function(X_p4)
idx_p4 = np.argsort(scores_p4[:, CLASSI.index('P4')])[-3:]
esempi_p4 = df_p4.iloc[idx_p4][['oggetto', 'descrizione_pulita', 'priorita_finale']]
```

### B. Calibrare la soglia di confidenza
Trovare la soglia ottimale con un validation set dedicato:
```python
soglie = np.arange(0.5, 0.95, 0.05)
# Per ogni soglia: valuta (costo_chiamate_LLM, macro_F1_risultante)
# Scegli la soglia che massimizza F1 con il minor numero di chiamate LLM
```

### C. Aggiungere la conversazione come feature
Il dataset completo (59k ticket) ha la `conversazione` con i messaggi cliente+operatore.
Aggiungere i primi 2 messaggi del cliente al testo input migliora il contesto per l'LLM.

### D. Endpoint FastAPI
```python
# Struttura endpoint suggerita
# POST /api/v1/classify/priority
# Body: {oggetto, descrizione, priorita_iniziale_cliente}
# Response: {priorita, confidenza, metodo, reasoning?}
```